#  Prenositeľnosť kódu 

Vďaka intenzívnemu využívaniu technológie HAL (Hardware Abstraction Layer) umožnuje pre MicroPython prenositeľnosť kódu nielen medzi mikrokontrolérmi v rámci jednej platformy, ale aj naprieč celým spektrom zariadení, na ktorých je portovaný. V nasledujúcom jednoduchom programe sú načítané dáta z teplomera a teplotného komparátora LM92 [6] pripojeného na zbernicu I2C číslo jedna. Dáta sú načítané ako 2 Byte z TEMPERATURE REGISTER na adrese čislo 0, čip obsahuje 7 registrov pre čítanie dát a nastavenie prahových hodnôt komparátorov a hysterézy.

```{code-block} python
:caption: Demo program
import machine

def read_LM92_TR(ic, addr):
    raw = ic.readfrom_mem(addr, 0, 2)  # nacitanie 2 byte z registra 0
    data = (raw[0] << 8) + raw[1]      # konverzia 16 bit format
    td = data >> 3                     # 2'compl b15-b3 temperature
    TEMP = (-(td & 0x1000) | (td & 0xFFF))* 0.0625
    L = data & 0x01                    # T_LOW  -> H, TEMP < 10 deg
    H = (data & 0x02) >> 1             # T_HIGH -> H  TEMP > 64 deg
    C = (data & 0x04) >> 2             # T_CRIT -> H  TEMP > 80 deg
    return TEMP, C, H, L
                                       # MCU STM32L432KC
ic=machine.I2C(1)                      # init I2C interface, PA10-SDA, PA9-SCL
print(ic.scan())                       # -> [75]  list of all devices in I2C(1)
print(read_LM92_TR(ic, 75))            # -> (24.5625, 0, 0, 0)
```

Uvedený program bude plne funkčný nielen na všetkých podporovaných mikrokontroléroch z triedy **STM32**, ale aj na ktomkoľvek zariadení, na ktoré je portovaný MikroPython a ktoré na palube obsahuje I2C rozhranie. Ak sú preto na vývoj periférneho zariadenia kladené vyššie požiadavky, môžeme samotný vývoj realizovať na procesore vyššej triedy a pre finálnu aplikáciu využiť jednoduchší procesor.
Modularita

Aby nebolo potrebné po resete mikrokontroléra opakovane nahrávať časti odladeného kódu do prostredia MicroPython-u, tento umožňuje pridanie nového kódu (frozen module) ako knižnice, ktorá sa stane súčasťou firmware. Celý postup je veľmi jednoduchý a spočíva v uložení súboru s kódom knižnice v Pythone do adresáru ./modules a následnom skompilovaní firmware a jeho nahratí do mikrokontroléra. Knižnica je dostupná pomocou štandardneho príkazu import.

Pri mikrokontroléroch s väčšou FLASH pamäťou je možné jej voľnú časť využiť ako pamäťové médium mapované ako file systém. Knižnica os poskutuje základné funkcie pre vytváranie adresárov a manipuláciu so súbormi. File systém je dostupný aj mimo prostredia MicroPython-u, pomocou utility pyboard je možné do neho ukladať, mazať súbory a vyvárať adresárovú štruktúru.
Flexibilita vývojovej platformy a škálovateľnosť

Elementárna verzia ``MikroPython`` bez knižníc má po skompilovaní veľkosť asi 20KB, pri konfigurácii pre jednotlivé platformy sa ale tvorcovia smažili optimálne využiť flash pamäte mikrokontrolérov doplnením čo najväčšieho množstva štandardných knižníc a podporou periférií. V prípade, že mikrokontrolér obsahuje väčšiu pamäť FLASH a RAM, je súčasťou firmware možnosť mapovania pamäte ako súborového systému, do ktorého je možné zaradiť aj externé pamäťové média ako je SD karta a pod. V prípade, že pri vývoji nie sú niektoré knižnice a drivery potrebné, je možné konfiguráciu firmware upraviť podľa aktuálnej potreby. MicroPython pre konfigurácou cieľového firmware pre každú platformu používa súbor mpconfigboard.h. Tento obsahuje premenné, pomocou ktorých vieme zaradiť alebo vyradiť z kompilácie firmware zvolené časti zdrojového kódu.
Rozširiteľnosť
Pri vývoji neštandardných periférií nemusí dostupná podpora periférií, ktoré sú súčasťou MicroPython-u, vyhovovať a môže vzniknúť požiadavka na nízkoúrovňovú obsluhu periférie. Podobne ako pri štandardnom Pythone, je možné aj MocroPython rozširovať o natívne moduly napísané v C/C++ a ktoré môžu využívať aj systémové knižnice pre obsluhu periférií mikrokontroléra. Pre vytvorenie modulu musíme rovnako ako v štandardnom Pythone vytvoriť rozhranie medzi natívnym modulom a jeho reprezentáciou v Pythone. Pre štandardný Python existujú generátory kódu, napr. SWIG, ktoré vygenerujú potrebné rozhranie na základe zdrojového kódu modulu. Pri MicroPython-e sa používa opačný postup, môžeme použiť generátor rozhrania (stub), ktorý vytvorí stub pozostávajúci z makier v C na základe deklarácie funkcie v Pythone.